# Explicit Fabric IQ orchestration with delegated authentication


In [ ]:
cd "$(git rev-parse --show-toplevel)" && printf "Repository: %s\n" "$PWD"

## Running this notebook

Select the Bash kernel from the repository's `.venv` environment. Code cells
use the Shell Script language and share one Bash session.

Sign in from a terminal before running the notebook:

```bash
az login --tenant f71980b2-590a-4de9-90d5-6fbc867da951
```

Requires Azure CLI, jq and HTTPie. Run cells from top to bottom. Run All reads
MCP metadata and starts one real Logic App request for dataset case B-006,
which can incur inference costs. Re-running the POST cell starts another run.
Deployment, activation and the alternative Python client remain manual examples.
The polling cell waits for the current run rather than starting another one.
Tokens stay in kernel memory; don't print them or enable shell tracing.



## What the workflow does

The [Consumption definition](../logicapps/ontology-orchestration/workflow.json)
calls the Fabric IQ ontology MCP endpoint directly, without invoking the Foundry
IQ agent:

1. Call `list_ontology_entities` without name filters to retrieve all entities,
   including properties, synonyms and source links.
2. Call `list_ontology_rules` without entity filters to retrieve every rule,
   including rules that would be missed by entity-only filtering.
3. Give this metadata and the complete original question to a Foundry model.
   Resolve entity references and produce an ontology-grounded query rewrite.
   Record relevance and a reason for every rule. A fact-dependent rule remains
   potentially applicable until evidence has been retrieved.
4. Verify entity names, exact rule coverage, unresolved semantics and request
   length. Send the unchanged original question plus the clarification to
   `ask_ontology` as its `request` argument.
5. Let Fabric IQ retrieve evidence through the ontology's bindings, including
   the semantic model where applicable. The workflow does not select SQL tables,
   reproduce semantic measures or substitute its own retrieval implementation.
6. Independently validate the returned answer against the original question,
   the rewrite, ontology entities and every rule. Return the original Fabric IQ
   answer only after successful model review and structural checks.

The [stage prompts](../logicapps/ontology-orchestration/prompts.json) contain
orchestration instructions, not business policies. Policies remain in the
ontology. An empty rule set is supported; nonempty sets require exact coverage.
There is no physical SQL connector, definition REST/TMDL download, tool-selection
agent, order placement or notification action in this workflow.


## Fabric IQ tools list


In [1]:
MCP_URL='https://api.fabric.microsoft.com/v1/mcp/dataPlane/workspaces/66d9c2b2-d35e-4555-9e74-926164ce584c/items/8750de7f-d46f-4dd4-84f4-a7e31f68ff24/ontologyEndpoint'

FABRIC_IQ_MCP=$(az rest --method post \
  --resource https://api.fabric.microsoft.com \
  --url "$MCP_URL" \
  --headers 'Content-Type=application/json' \
    'Accept=application/json, text/event-stream' \
  --body '{"jsonrpc":"2.0","id":"tools","method":"tools/list","params":{}}' -o json) &&
jq '.result.tools | map({
  name,
  description,
  parameters: (.inputSchema.properties | to_entries | map({
    name: .key,
    description: .value.description
  }))
})' <<< "$FABRIC_IQ_MCP"


[
  {
    "name": "ask_ontology",
    "description": "Answers day-to-day business questions using an established Fabric ontology and its bound data, or generates grounded, validated, runnable code. For code requests, include the required `queryLanguage` and execution target inside the `request`. The requested artifact can be native KQL, SQL, DAX, cloud-runnable Python, a DSL definition, or Ontology or Eventhouse-runnable KQL that executes and joins KQL, SQL, and DAX sources. The tool performs read-only operations and returns either an answer with supporting evidence or validated runnable code with grounding and validation details.",
    "parameters": [
      {
        "name": "request",
        "description": "The business question or requested artifact. Include the relevant scope, filters, time range, and desired output, such as an answer, data, or reusable code. For code requests, include `queryLanguage` and the intended execution target in the prompt."
      }
    ]
  },
  {
    "na


## Fabric IQ list_ontology_entities


In [ ]:
WORKSPACE_ID='66d9c2b2-d35e-4555-9e74-926164ce584c'
ONTOLOGY_ID='8750de7f-d46f-4dd4-84f4-a7e31f68ff24'
MCP_URL="https://api.fabric.microsoft.com/v1/mcp/dataPlane"
MCP_URL+="/workspaces/${WORKSPACE_ID}/items/${ONTOLOGY_ID}/ontologyEndpoint"

FABRIC_IQ_MCP_LIST_RESULT=$(az rest --method post \
  --resource https://api.fabric.microsoft.com \
  --url "$MCP_URL" \
  --headers 'Content-Type=application/json' \
    'Accept=application/json, text/event-stream' \
  --body '{
    "jsonrpc": "2.0",
    "id": "entities",
    "method": "tools/call",
    "params": {
      "name": "list_ontology_entities",
      "arguments": {}
    }
  }' -o json) &&
echo "$FABRIC_IQ_MCP_LIST_RESULT" | jq '.result.structuredContent.values | map({name, synonyms})'


In [ ]:
FABRIC_IQ_MCP_LIST_RESULT=$(az rest --method post \
  --resource https://api.fabric.microsoft.com \
  --url "$MCP_URL" \
  --headers 'Content-Type=application/json' \
    'Accept=application/json, text/event-stream' \
  --body '{
    "jsonrpc": "2.0",
    "id": "entities",
    "method": "tools/call",
    "params": {
      "name": "list_ontology_entities",
      "arguments": {"entityName": "qwert"}
    }
  }' -o json) &&
echo "$FABRIC_IQ_MCP_LIST_RESULT" | jq '.result.structuredContent.values | map({name, synonyms})'


## Anfrage an Logic App Workflow

The success response contains only `response`, `validation` and `runId`.
The returned answer is the joined text content of the actual `ask_ontology`
result, not a locally generated replacement. Entity metadata, rule definitions,
the rewrite and the raw tool result remain internal; their checks still run.
Model validation is not independent verification of the underlying data.

Use `runId` to find the workflow run in Azure. Run history retains action status,
timing and failure location, subject to the configured retention. Secure inputs
and outputs remain hidden, so this is not a full content trace. Reducing the
response does not add Foundry/OpenTelemetry tracing or expose Fabric's internal
queries. The manual client validates the compact result; it no longer
cross-checks the internal metadata or raw answer, which are not returned.


In [ ]:
SUBSCRIPTION='98525264-1eb4-493f-983d-16a330caa7f6'
RESOURCE_ID="$(az logic workflow show --subscription "$SUBSCRIPTION" \
  --resource-group rg-projplan-demo --name logic-projplan-tbv74x \
  --query id -o tsv)"

CALLBACK="$(az rest --method post \
  --url "https://management.azure.com${RESOURCE_ID}/triggers/\
When_a_request_is_received/listCallbackUrl?api-version=2019-05-01" \
  --query value -o tsv)"

ARM_TOKEN="$(az account get-access-token \
  --resource https://management.azure.com/ --query accessToken -o tsv)"

FABRIC_TOKEN="$(az account get-access-token \
  --resource https://api.fabric.microsoft.com --query accessToken -o tsv)"

QUESTION_B_006='Gib mir eine Liste aller qwert mit ihrer ID und ihrem Namen.'


In [8]:
WORKFLOW_HTTP_RESULT="$(http --check-status --ignore-stdin --pretty=none --print=hb --json POST "$CALLBACK" "Authorization:@"<(printf 'Bearer %s' "$ARM_TOKEN") query="$QUESTION_B_006" "fabricAccessToken=@"<(printf '%s' "$FABRIC_TOKEN"))"
printf '%s\n' "$WORKFLOW_HTTP_RESULT" | sed '1,/^[[:space:]]*$/d' | jq .


{
  "properties": {
    "waitEndTime": "2026-11-16T19:08:35.0630087Z",
    "startTime": "2026-11-16T19:08:35.0630087Z",
    "status": "Running",
    "correlation": {
      "clientTrackingId": "08584067507704181973637678100CU23"
    },
    "workflow": {
      "id": "/workflows/d94b1945eaa34587ba79c76b84cdd22f/versions/08584067513061605737",
      "name": "08584067513061605737",
      "type": "Microsoft.Logic/workflows/versions"
    },
    "trigger": {
      "name": "When_a_request_is_received",
      "inputsLink": {
        "contentSize": 267,
        "secureData": {
          "properties": [
            "inputs"
          ]
        }
      },
      "outputsLink": {
        "contentSize": 3655,
        "secureData": {
          "properties": [
            "outputs"
          ]
        }
      },
      "startTime": "2026-11-16T19:08:35.0480475Z",
      "endTime": "2026-11-16T19:08:35.0480475Z",
      "originHistoryName": "08584067507704181973637678100CU23",
      "correlation": {
       

In [10]:
POLL_URL="$(awk 'tolower($1) == "location:" {
    sub(/\r$/, "", $2); print $2; exit
  }' <<< "$WORKFLOW_HTTP_RESULT")"

WORKFLOW_HTTP_RESULT="$(http --check-status --ignore-stdin --pretty=none --print=hb GET "$POLL_URL" "Authorization:@"<(printf 'Bearer %s' "$ARM_TOKEN"))" &&
printf '%s\n' "$WORKFLOW_HTTP_RESULT" | sed '1,/^[[:space:]]*$/d' | jq .


{
  "properties": {
    "waitEndTime": "2026-11-16T19:08:35.0630087Z",
    "startTime": "2026-11-16T19:08:35.0630087Z",
    "status": "Running",
    "correlation": {
      "clientTrackingId": "08584067507704181973637678100CU23"
    },
    "workflow": {
      "id": "/workflows/d94b1945eaa34587ba79c76b84cdd22f/versions/08584067513061605737",
      "name": "08584067513061605737",
      "type": "Microsoft.Logic/workflows/versions"
    },
    "trigger": {
      "name": "When_a_request_is_received",
      "inputsLink": {
        "contentSize": 267,
        "secureData": {
          "properties": [
            "inputs"
          ]
        }
      },
      "outputsLink": {
        "contentSize": 3655,
        "secureData": {
          "properties": [
            "outputs"
          ]
        }
      },
      "startTime": "2026-11-16T19:08:35.0480475Z",
      "endTime": "2026-11-16T19:08:35.0480475Z",
      "originHistoryName": "08584067507704181973637678100CU23",
      "correlation": {
       


## Errors and diagnostic stages

The run's `stage` and action names identify the failure location:

* `fabric-iq-metadata`: `List_ontology_entities`, `List_ontology_rules`,
  metadata validation or parsing
* `entity-matching-and-query-rewrite`: `Rewrite_query` and `Check_rewrite`
* `ask-ontology`: `Ask_ontology` and `Check_ontology_result`
* `answer-validation`: `Validate_answer` and `Check_validation`

HTTP transport failures, including Fabric 401/403, return HTTP 502 with the
stage and run ID. JSON-RPC/tool failures and invalid metadata, rewrites or
answers return explicit HTTP 422 errors. Failed runs never return a
success-shaped assessment.

The combined original question, instructions and clarification must fit
`ask_ontology`'s 10000-character request limit. Overlong input is rejected,
not truncated. Each metadata response and the ontology answer are bounded
to 120000 characters. Unsupported response formats fail rather than falling
back to direct SQL.

## Historical SQL comparison

The previous REST/SQL implementation passed a smoke test but only 1/6 diagnostic
cases. IQ v8 passed 6/6 using the same evaluator v7 and reference dataset.
Four workflow cases were rejected during source/rule screening and one failed
SQL login. That comparison changed the retrieval implementation as well as
orchestration, so it did not isolate the effect of explicit ordering.

Captures and judgments remain in the ignored
`evaluations/manual_runs/explicit-comparison-20261116/` directory.
Those scores do not evaluate this new MCP-based workflow. Its answer-quality
tests are intentionally left to the user's delegated manual execution.

## Microsoft references

* [Ontology MCP tools and authentication](https://learn.microsoft.com/en-us/fabric/iq/ontology/how-to-use-ontology-mcp-server)
* [Managed Identity for Foundry HTTP calls](https://learn.microsoft.com/en-us/azure/logic-apps/authenticate-with-managed-identity)
* [Workflow security and secure run history](https://learn.microsoft.com/en-us/azure/logic-apps/set-up-security-permissions)
* [Consumption workflow resource](https://learn.microsoft.com/en-us/azure/templates/microsoft.logic/2019-05-01/workflows)
